In [1]:
import numpy as np
import glob
import gensim
import gensim.corpora as corpora
from gensim.utils import simple_preprocess
from gensim.models import CoherenceModel

#spacy
import spacy
from nltk.corpus import stopwords

#vis
import pyLDAvis
import pyLDAvis.gensim_models

import json
from datetime import datetime, timezone
import pandas as pd
import csv
import matplotlib.pyplot as plt
import nltk

import io

In [2]:
documents = pd.read_csv('../Corpus_data/sa_corpus.csv')

In [3]:
docs  = documents['postdoc'].to_list()

In [4]:
# Tokenize the documents.
from nltk.tokenize import RegexpTokenizer

# Split the documents into tokens.
tokenizer = RegexpTokenizer(r'\w+')
for idx in range(len(docs)):
    docs[idx] = docs[idx].lower()  # Convert to lowercase.
    docs[idx] = tokenizer.tokenize(docs[idx])  # Split into words.

# Remove numbers, but not words that contain numbers.
docs = [[token for token in doc if not token.isnumeric()] for doc in docs]

# Remove words that are only one character.
docs = [[token for token in doc if len(token) > 1] for doc in docs]

In [5]:
# Lemmatize the documents.
from nltk.stem.wordnet import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()
docs = [[lemmatizer.lemmatize(token) for token in doc] for doc in docs]

In [6]:
mallard_stopwords = set()
#stopwords from mallard website
with open('en.txt','r') as f:
    num = 0
    for line in f:
        mallard_stopwords.add(line.strip())
#Removed tailored stopwords found after previous runs.
mallard_stopwords.update(['ve','ll','doe','don','ha','wa','didn','bos','im','ive','doesn','lot','wasn','people','feel','thing','feeling']) #'anxiety','social'
docs = [[token for token in doc if token not in mallard_stopwords] for doc in docs]

In [7]:
# Remove rare and common tokens.
from gensim.corpora import Dictionary
dictionary = Dictionary(docs)
# Filter out words that occur less than 20 documents, or more than 90% of the documents.
dictionary.filter_extremes(no_below=20, no_above=0.9)

In [8]:
# Bag-of-words representation of the documents.
corpus = [dictionary.doc2bow(doc) for doc in docs]

In [9]:
print('Number of unique tokens: %d' % len(dictionary))
print('Number of documents: %d' % len(corpus))

Number of unique tokens: 7163
Number of documents: 27197


In [10]:
# Train LDA model.
from gensim.models import LdaModel

# Set training parameters.
num_topics = 13
chunksize = 4096
passes = 140
iterations = 2800
eval_every = 0  # Don't evaluate model perplexity, takes too much time.
decay = 0.5
offset = 64
# Make an index to word dictionary.
temp = dictionary[0]  # This is only to "load" the dictionary.
id2word = dictionary
model = LdaModel(
    corpus=corpus,
    id2word=id2word,
    chunksize=chunksize,
    alpha='auto',
    eta='auto',
    iterations=iterations,
    num_topics=num_topics,
    passes=passes,
    eval_every=eval_every,
    decay=decay,
    offset=offset
)

In [16]:
top_topics = model.top_topics(corpus)

# Average topic coherence is the sum of topic coherences of all topics, divided by the number of topics.
avg_topic_coherence = sum([t[1] for t in top_topics]) / num_topics
print('Average topic coherence: %.4f.' % avg_topic_coherence)

Average topic coherence: -1.8541.


In [17]:
import pyLDAvis
import pyLDAvis.gensim_models

pyLDAvis.enable_notebook()

vis = pyLDAvis.gensim_models.prepare(
    topic_model=model,
    corpus=corpus,
    dictionary=dictionary
)

vis

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
6      0.094242 -0.020273       1        1  20.969731
8      0.153894 -0.034409       2        1  18.420785
12     0.228909 -0.024160       3        1  15.622893
4      0.056639  0.047786       4        1  14.315581
1     -0.047307 -0.175817       5        1   7.166284
11     0.158966 -0.129590       6        1   4.165000
9     -0.114884 -0.010245       7        1   3.708193
2     -0.000772 -0.007957       8        1   3.522373
10     0.120871  0.246643       9        1   3.324307
0     -0.140537  0.331108      10        1   2.576476
3     -0.211198 -0.126647      11        1   2.137222
7     -0.104015 -0.084629      12        1   2.108110
5     -0.194808 -0.011808      13        1   1.963044, topic_info=         Term          Freq         Total Category  logprob  loglift
361    friend  36084.000000  36084.000000  Default  30.0000  30.0000
11    anxiety  53012.000000  53012.000000  Default  29.0000  29.0000
202    social  41796.000000  41796.000000  Default  28.0000  28.0000
643       job  12823.000000  12823.000000  Default  27.0000  27.0000
309    school  13739.000000  13739.000000  Default  26.0000  26.0000
...       ...           ...           ...      ...      ...      ...
469     email    534.448627   1074.728182  Topic13  -4.6782   3.2321
1411    write    559.689762   1925.528757  Topic13  -4.6321   2.6951
175      read    692.436996   3892.701590  Topic13  -4.4193   2.2040
1139  youtube    368.400743   1057.957401  Topic13  -5.0503   2.8758
717   writing    363.597307   1335.558090  Topic13  -5.0634   2.6296

[896 rows x 6 columns], token_table=      Topic      Freq    Term
term                         
1757      7  0.992613     6th
3081      2  0.047250     8th
3081      7  0.952877     8th
5370      4  0.988484     abt
2034      2  0.782582  accept
...     ...       ...     ...
3157      6  0.996078  zoloft
834       3  0.532568    zone
834       5  0.010087    zone
834      12  0.456583    zone
3933      7  0.999159    zoom

[2143 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[7, 9, 13, 5, 2, 12, 10, 3, 11, 1, 4, 8, 6])

In [14]:
from gensim.test.utils import datapath
# Save model to disk.
model.save('lda.model')